# 红薯分类

1）导入需要的模块

In [6]:
import torch # 导入最基本的张量模块
import torchvision #包含各种数据集
import torch.nn as nn #包含经典网络
import torch.optim as optim #优化器
import torch.nn.functional as F #函数如交叉熵损失
import torchvision.transforms as transforms #数据预处理模块
from torch.utils.tensorboard import SummaryWriter # 数据可视化

In [7]:
from torch.utils.data import Dataset # 导出加载数据集的模块
from PIL import Image # 加载图像的模块
import os # 对文件进行操作的模块
from torchvision.datasets import ImageFolder

2）导入数据集，由于已经按照文件夹分好类，所以可以使用ImageFolder来加载

In [8]:
train_root='E:\HongShu\HongShu' 
    # 如果要使用预训练模型，那么数据预处理方法应该和预训练模型的数据预处理一样，
    # 这里显然和预训练模型的预处理不一致
"""
#这是之前和预训练模型不一致的预处理方式，
#     transforms.Resize(224),
# #     transforms.RandomResizedCrop(224,scale=(0.6,1.0),ratio=(0.8,1.0)),
#     transforms.RandomHorizontalFlip(p=0.5),
#     torchvision.transforms.RandomVerticalFlip(p=0.5),
#     torchvision.transforms.RandomGrayscale(p=0.2),

# #     torchvision.transforms.ColorJitter(brightness=0.5, contrast=0, saturation=0, hue=0),
# #     torchvision.transforms.ColorJitter(brightness=0, contrast=0.5, saturation=0, hue=0),
#     transforms.ToTensor(),
#     transforms.Normalize(mean=[.5, .5, .5], std=[.5, .5, .5])
""" 
train_transform=transforms.Compose([
    transforms.Resize(224),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),    
])

all_data=torchvision.datasets.ImageFolder(
    root=train_root,
    transform=train_transform
)

train_set=torch.utils.data.DataLoader(
    all_data,
    batch_size=10,
    shuffle=True
)

batch=next(iter(train_set)) #iter生成迭代器，next得到下一个对象
images,labels = batch #得到一个批次的图像和标签

In [9]:
print(all_data.classes)
print(all_data.class_to_idx)

['JiSun', 'WanZheng']
{'JiSun': 0, 'WanZheng': 1}


3）定义网络结构

In [10]:
resnet18=torchvision.models.resnet18(pretrained=True) #直接加载预训练模型
# print(resnet18) # 打印预训练模型结构

for param in resnet18.parameters(): # 循环遍历网络中的参数，
    param.requires_grad=False # 冻结参数 
resnet18.fc=torch.nn.Linear(512,2) # 将线性层最后的输出改为2，用于2分类
# 定义新的全连接层之后，全连接层中的parma。requis_grad参数会被默认设置为True，
# 所以不需要再次遍历参数来进行解冻操作。

# print(resnet18) # 打印修改之后的网络结构模型

In [12]:
learning_rate=0.00001 # 定义一个学习率，便于更改
optimizer=optim.Adam(resnet18.fc.parameters(),lr=learning_rate) #使用Adam优化函数，需要传入网络的参数和学习率
# optimizer.step() #参数更新

#此处定义一个计算预测正确的数目的函数
def get_num_correct(preds,labels):
    return preds.argmax(dim=1).eq(labels).sum().item() #argmax返回指定维度最大值的索引

device=torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
resnet18=resnet18.to(device) # 将网络移到GPU上
images=images.to(device) # 将图像和标签也传入GPU，转换为GPU格式的张量，加速计算
labels=labels.to(device)

# 分批次训练
for epoch in range(20):
    total_loss=0
    total_correct=0
    for batch in train_set:
#         images,labels=batch
        preds=resnet18(images) #将图像传入网络得到预测输出
        loss=F.cross_entropy(preds,labels) #使用交叉熵损失函数得到损失
        optimizer.zero_grad() #将梯度归零，否则梯度会累积
        
        loss=loss.to(device) #将损失传入GPU，变为GPU格式
        
        loss.backward() #反向传播
        optimizer.step() #使用优化器更新权重参数
        
        total_loss+=loss.item() #总损失,需要将张量变为可以数值计算的浮点型
        
        total_correct+=get_num_correct(preds,labels) #预测正确的个数
        
# 实例化tensorboard
    tb=SummaryWriter('E:/run/HongShu/2') #指定可视化日志文件的存储路径        
    tb.add_scalar('loss',total_loss,epoch) #将标量total_loss加载到日志文件，使其可视化
    tb.add_scalar('correc numbers',total_correct,epoch)
    tb.add_scalar('accuracy',total_correct/len(all_data),epoch) 

        #加入网络的结构图，需要传入的是网络名称和每个批次处理的图像，
        #图像维度为（batch_size,channels，height，width），即批次*通道*高度*宽度
    tb.add_graph(resnet18,images) 

#     tb.add_histogram('conv1.bias',resnet18.conv1.bias,epoch) #偏置图
#     tb.add_histogram('conv1.weight',resnet18.conv1.weight,epoch) #权重图
#     tb.add_histogram('conv1.weight.grad',resnet18.conv1.weight.grad,epoch) #权重梯度图
    tb.close() #关闭日志文件
    
    print('epoch',epoch,'total_correct:',total_correct,'total_loss:',total_loss,
         'accuracy',total_correct/len(all_data))

epoch 0 total_correct: 460 total_loss: 11.658092141151428 accuracy 1.0
epoch 1 total_correct: 460 total_loss: 11.06755055487156 accuracy 1.0
epoch 2 total_correct: 460 total_loss: 10.516519844532013 accuracy 1.0
epoch 3 total_correct: 460 total_loss: 10.001691117882729 accuracy 1.0
epoch 4 total_correct: 460 total_loss: 9.519996911287308 accuracy 1.0
epoch 5 total_correct: 460 total_loss: 9.068744644522667 accuracy 1.0
epoch 6 total_correct: 460 total_loss: 8.645556345582008 accuracy 1.0
epoch 7 total_correct: 460 total_loss: 8.248304054141045 accuracy 1.0
epoch 8 total_correct: 460 total_loss: 7.875062271952629 accuracy 1.0
epoch 9 total_correct: 460 total_loss: 7.524077698588371 accuracy 1.0
epoch 10 total_correct: 460 total_loss: 7.193746283650398 accuracy 1.0
epoch 11 total_correct: 460 total_loss: 6.882595285773277 accuracy 1.0
epoch 12 total_correct: 460 total_loss: 6.589272394776344 accuracy 1.0
epoch 13 total_correct: 460 total_loss: 6.312532484531403 accuracy 1.0
epoch 14 tota

In [ ]:
# model = torch.load("vgg16_model1.pth") # 下载好之后加载模型